In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 7 - WEEK 9 BAYESIAN OPTIMISATION
# Run from inside the week9/ folder
# ============================================================

# ------------------------------------------------------------
# 1. Load cumulative Week 9 data
# ------------------------------------------------------------

X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 8 calibration check
# ------------------------------------------------------------
#
# Week 8 selected:
# [0.146395, 0.328545, 0.399878,
#  0.246398, 0.271970, 0.672216]
#
# GP prediction:
# mean ≈ 2.684007
# std  ≈ 0.107590
#
# Actual:
# 2.7916791693435754
# ------------------------------------------------------------

week8_pred_mean = 2.684007493398089
week8_pred_std = 0.10758970794809386
week8_actual = 2.7916791693435754

week8_error = week8_actual - week8_pred_mean
week8_z_error = week8_error / week8_pred_std

print("\n================================")
print("WEEK 8 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week8_pred_mean)
print("Predicted std :", week8_pred_std)
print("Actual        :", week8_actual)

print("\nPrediction error:")
print(week8_error)

print("\nError / predicted std:")
print(week8_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(6) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(120000, 6)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(80000, 6)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(200000, 6)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

# Remove near-duplicates

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 6. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 7. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 8. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 9. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )

X shape: (38, 6)
Y shape: (38,)

Current best:
[0.146395 0.328545 0.399878 0.246398 0.27197  0.672216] -> 2.7916791693435754

Y range:
min = 0.0027014650245082332
max = 2.7916791693435754
std = 0.8865332263279528

WEEK 8 CALIBRATION CHECK
Predicted mean: 2.684007493398089
Predicted std : 0.10758970794809386
Actual        : 2.7916791693435754

Prediction error:
0.10767167594548654

Error / predicted std:
1.0007618572348225


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.616**2 * Matern(length_scale=[0.399, 2, 2, 0.191, 0.176, 0.399], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[0.39870089 2.         2.         0.19121207 0.17575738 0.3989461 ]

Normalised inverse-lengthscale sensitivity:
[0.14811121 0.02952604 0.02952604 0.30883025 0.33598629 0.14802017]

Local widths: [0.09967522 0.1        0.1        0.04780302 0.04393935 0.09973652]
Wide widths: [0.19935044 0.2        0.2        0.09560603 0.08787869 0.19947305]

Candidates after duplicate filtering:
400000

PRIMARY EI
candidate = [0.16522977 0.22799513 0.58728299 0.25148122 0.27764608 0.67443723]
mean = 2.7797683614352584
std = 0.07842338624398089
EI = 0.025691150425521565

HIGHEST PREDICTED MEAN
candidate = [0.15306927 0.3042981  0.47637588 0.24944912 0.266346   0.67815383]
mean = 2.7937807488084143
std = 0.03000324410049695

UCB DIAGNOSTICS

beta=0.1 
 candidate = [0.15306927 0.3042981  0.47637588 0.24944912 0.266346   0.67815383] 
 mean = 2.793781 
 std

In [2]:
# ============================================================
# FINAL FUNCTION 7 - WEEK 9 SELECTION
# ============================================================
#
# Week 8 calibration error was approximately +1.00 sigma.
# Moderate exploration performed well and produced the
# current best observation.
#
# beta = 0.5 again provides a useful exploration-exploitation
# balance while remaining in the same broad high-value region.

beta = 0.5

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week9_candidate = candidates[final_idx]

print("Week 9 Function 7 candidate:")
print(week9_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week9_candidate
)

print("\nPortal format:")
print(portal)

Week 9 Function 7 candidate:
[0.16522977 0.22799513 0.58728299 0.25148122 0.27764608 0.67443723]

Predicted mean:
2.7797683614352584

Predicted std:
0.07842338624398089

UCB:
2.8189800545572488

Portal format:
0.165230-0.227995-0.587283-0.251481-0.277646-0.674437
